In [26]:
# Exploratory Data Analysis

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from scipy.stats import skew
from scipy.stats import kurtosis

# -------------------------
# Display Options
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", "{:,.2f}".format)

# Plotly Template
template = "plotly_white"

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [27]:
# Load cleaned dataset

df = pd.read_csv("../Data/shipments_cleaned.csv")

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [28]:
df.head()

,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status,missing_booking_date,missing_pickup_date,missing_actual_delivery,invalid_date_sequence,delivery_delay_days,transit_days,booking_to_pickup_days,cost_per_km,delivery_performance,on_time_flag,analysis_ready,delay_category,distance_band
0,SHP003604,2026-04-18,2026-04-21,2026-04-27,Chandigarh,Guwahati,North,PTL,CARR_07,CUST_085,2026-04-27,2026-04-29,"72,921.16",894,Delivered,False,False,False,False,2.00,8.00,3.00,81.57,Late,0.00,True,1–2 Days Late,501–1000 km
1,SHP003362,2026-06-01,2026-06-01,2026-06-04,Kolkata,Raipur,East,FTL,CARR_03,CUST_108,2026-06-04,2026-06-03,"2,451.90",91,Delivered,False,False,False,False,-1.00,2.00,0.00,26.94,On Time,1.00,True,Early,0–500 km
2,SHP000796,2026-01-17,2026-01-20,2026-01-23,Surat,Jaipur,West,FTL,CARR_05,CUST_108,2026-01-23,NaN,"10,320.66",395,In-Transit,False,False,True,False,NaN,NaN,3.00,26.13,In Transit,NaN,False,NaN,0–500 km
3,SHP000291,2026-04-25,2026-04-25,2026-05-01,Nagpur,Bhubaneswar,Central,FTL,CARR_09,CUST_063,2026-05-01,2026-05-02,"30,242.52",1223,Delayed,False,False,False,False,1.00,7.00,0.00,24.73,Late,0.00,True,1–2 Days Late,1001–1500 km
4,SHP003739,2026-01-29,2026-01-30,2026-02-03,Ahmedabad,Hyderabad,West,LTL,CARR_04,CUST_064,2026-02-03,2026-02-01,"18,434.26",1664,Delivered,False,False,False,False,-2.00,2.00,1.00,11.08,On Time,1.00,True,Early,1501–2000 km


In [29]:
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Rows    : 5,000
Columns : 28


In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   shipment_id              5000 non-null   str    
 1   booking_date             4929 non-null   str    
 2   pickup_date              4913 non-null   str    
 3   delivery_date            5000 non-null   str    
 4   origin_city              5000 non-null   str    
 5   destination_city         5000 non-null   str    
 6   region                   5000 non-null   str    
 7   mode                     5000 non-null   str    
 8   carrier_id               5000 non-null   str    
 9   customer_id              5000 non-null   str    
 10  promised_delivery_date   5000 non-null   str    
 11  actual_delivery_date     3518 non-null   str    
 12  freight_cost             5000 non-null   float64
 13  distance_km              5000 non-null   int64  
 14  status                   5000 non-n

In [31]:
date_columns = [
    "booking_date",
    "pickup_date",
    "delivery_date",
    "promised_delivery_date",
    "actual_delivery_date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(df[date_columns].dtypes)

booking_date              datetime64[us]
pickup_date               datetime64[us]
delivery_date             datetime64[us]
promised_delivery_date    datetime64[us]
actual_delivery_date      datetime64[us]
dtype: object


In [32]:
#dataset dimensions
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"Number of Shipments : {df.shape[0]:,}")
print(f"Number of Columns   : {df.shape[1]}")

DATASET OVERVIEW
Number of Shipments : 5,000
Number of Columns   : 28


In [33]:
# column names
df.columns.tolist()

['shipment_id',
 'booking_date',
 'pickup_date',
 'delivery_date',
 'origin_city',
 'destination_city',
 'region',
 'mode',
 'carrier_id',
 'customer_id',
 'promised_delivery_date',
 'actual_delivery_date',
 'freight_cost',
 'distance_km',
 'status',
 'missing_booking_date',
 'missing_pickup_date',
 'missing_actual_delivery',
 'invalid_date_sequence',
 'delivery_delay_days',
 'transit_days',
 'booking_to_pickup_days',
 'cost_per_km',
 'delivery_performance',
 'on_time_flag',
 'analysis_ready',
 'delay_category',
 'distance_band']

In [34]:
# data types summary
dtype_summary = (
    df.dtypes
      .value_counts()
      .rename_axis("Data Type")
      .reset_index(name="Count")
)

dtype_summary

,Data Type,Count
0,str,11
1,float64,6
2,datetime64[us],5
3,bool,5
4,int64,1


In [35]:
# missing values summary
missing_summary = (
    df.isna()
      .sum()
      .to_frame("Missing Values")
)

missing_summary["Percentage"] = (
    missing_summary["Missing Values"]
    / len(df)
    * 100
).round(2)

missing_summary = (
    missing_summary
    .sort_values(
        "Missing Values",
        ascending=False
    )
)

missing_summary

,Missing Values,Percentage
transit_days,1545,30.90
actual_delivery_date,1482,29.64
delivery_delay_days,1482,29.64
on_time_flag,1482,29.64
delay_category,1482,29.64
booking_to_pickup_days,155,3.10
pickup_date,87,1.74
booking_date,71,1.42
delivery_date,0,0.00
origin_city,0,0.00


In [36]:
# analysis ready summary
analysis_summary = pd.DataFrame({

    "Metric": [

        "Total Shipments",

        "Analysis Ready",

        "Not Analysis Ready"

    ],

    "Count": [

        len(df),

        df["analysis_ready"].sum(),

        (~df["analysis_ready"]).sum()

    ]

})

analysis_summary["Percentage"] = (

    analysis_summary["Count"]

    / len(df)

    *100

).round(2)

analysis_summary

,Metric,Count,Percentage
0,Total Shipments,5000,100.00
1,Analysis Ready,3444,68.88
2,Not Analysis Ready,1556,31.12


In [37]:
# audit flags summary
audit_summary = pd.DataFrame({

    "Metric":[

        "Missing Booking",

        "Missing Pickup",

        "Missing Actual Delivery",

        "Invalid Date Sequence"

    ],

    "Count":[

        df["missing_booking_date"].sum(),

        df["missing_pickup_date"].sum(),

        df["missing_actual_delivery"].sum(),

        df["invalid_date_sequence"].sum()

    ]

})

audit_summary["Percentage"] = (

    audit_summary["Count"]

    / len(df)

    *100

).round(2)

audit_summary

,Metric,Count,Percentage
0,Missing Booking,71,1.42
1,Missing Pickup,87,1.74
2,Missing Actual Delivery,1482,29.64
3,Invalid Date Sequence,74,1.48


In [38]:
# delivery performance distribution
delivery_summary = (
    df["delivery_performance"]
      .value_counts(dropna=False)
      .rename_axis("Delivery Performance")
      .reset_index(name="Shipments")
)

delivery_summary["Percentage"] = (
    delivery_summary["Shipments"]
    / len(df)
    *100
).round(2)

delivery_summary

,Delivery Performance,Shipments,Percentage
0,On Time,1795,35.90
1,Late,1723,34.46
2,Unknown,682,13.64
3,In Transit,499,9.98
4,Cancelled,301,6.02


In [39]:
print("Missing Actual Delivery:", df["missing_actual_delivery"].sum())
print("Invalid Date Sequence:", df["invalid_date_sequence"].sum())

print("\nOverlap:")

overlap = (
    df["missing_actual_delivery"] &
    df["invalid_date_sequence"]
).sum()

print(overlap)

Missing Actual Delivery: 1482
Invalid Date Sequence: 74

Overlap:
0


In [40]:
before_booking = (
    df["actual_delivery_date"] < df["booking_date"]
)

before_pickup = (
    df["actual_delivery_date"] < df["pickup_date"]
)

print("Before Booking :", before_booking.sum())
print("Before Pickup  :", before_pickup.sum())

print("Both Violations:",
      (before_booking & before_pickup).sum())

Before Booking : 39
Before Pickup  : 72
Both Violations: 37


To ensure the engineered audit flags were internally consistent, two validation checks were performed. First, the analysis_ready flag was verified against the underlying business rules. All 1,482 shipments missing an actual delivery date and all 74 shipments with invalid chronological sequences were mutually exclusive, confirming that the final count of 3,444 analysis-ready shipments was calculated correctly. Second, the invalid date sequence flag was reconciled against its constituent business-rule violations. Of the 39 shipments delivered before their booking date and 72 delivered before their pickup date, 37 violated both rules, resulting in 74 unique shipments flagged for invalid chronology. These checks provide additional confidence that the engineered audit features accurately represent the underlying shipment data.

# Numerical & Distribution Analysis

In [41]:
numerical_columns = [
    "distance_km",
    "freight_cost",
    "transit_days",
    "delivery_delay_days",
    "booking_to_pickup_days",
    "cost_per_km"
]

In [42]:
# Comprehensive Numerical Summary

summary_stats = []

for col in numerical_columns:

    s = df[col].dropna()

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    summary_stats.append({

        "Variable": col,

        "Count": s.count(),

        "Missing": df[col].isna().sum(),

        "Mean": round(s.mean(),2),

        "Median": round(s.median(),2),

        "Std Dev": round(s.std(),2),

        "Minimum": round(s.min(),2),

        "Q1": round(q1,2),

        "Q3": round(q3,2),

        "Maximum": round(s.max(),2),

        "Range": round(s.max()-s.min(),2),

        "IQR": round(iqr,2),

        "CV (%)": round((s.std()/s.mean())*100,2),

        "Skewness": round(s.skew(),2),

        "Kurtosis": round(s.kurtosis(),2)

    })

summary_df = pd.DataFrame(summary_stats)

summary_df

,Variable,Count,Missing,Mean,Median,Std Dev,Minimum,Q1,Q3,Maximum,Range,IQR,CV (%),Skewness,Kurtosis
0,distance_km,5000,0,"1,279.76","1,280.00",711.88,50.00,655.00,"1,901.00","2,499.00","2,449.00","1,246.00",55.63,0.00,-1.22
1,freight_cost,5000,0,"33,741.78","18,014.10","65,530.71",412.31,"9,491.34","32,336.01","788,479.36","788,067.05","22,844.67",194.21,5.88,41.20
2,transit_days,3455,1545,5.33,5.00,3.05,-3.00,3.00,8.00,10.00,13.00,5.00,57.34,-0.14,-0.82
3,delivery_delay_days,3518,1482,0.28,0.00,3.65,-11.00,-2.00,3.00,8.00,19.00,5.00,"1,291.68",-0.14,-0.50
4,booking_to_pickup_days,4845,155,1.52,2.00,1.12,0.00,1.00,3.00,3.00,3.00,2.00,73.87,-0.02,-1.37
5,cost_per_km,5000,0,26.27,13.29,41.79,6.80,10.88,25.22,335.78,328.98,14.33,159.08,4.58,22.32


## Shipment Distance

In [43]:
distance = df["distance_km"]

distance.describe()

count   5,000.00
mean    1,279.76
std       711.88
min        50.00
25%       655.00
50%     1,280.00
75%     1,901.00
max     2,499.00
Name: distance_km, dtype: float64

In [44]:
fig = px.histogram(

    df,

    x="distance_km",

    nbins=40,

    title="Distribution of Shipment Distance",

    template="plotly_white"

)

fig.update_layout(

    xaxis_title="Distance (km)",

    yaxis_title="Number of Shipments"

)

fig.show()

In [45]:
fig = px.box(

    df,

    y="distance_km",

    title="Shipment Distance Boxplot",

    template="plotly_white"

)

fig.update_layout(

    yaxis_title="Distance (km)"

)

fig.show()

Shipment distance is approximately uniformly distributed across the observed range (50–2,499 km), with almost identical mean and median values. Unlike many logistics datasets where shipments cluster around shorter distances, this dataset represents a broad mix of short-, medium-, and long-haul transportation.

## Freight Cost

In [46]:
df["freight_cost"].describe()

count     5,000.00
mean     33,741.78
std      65,530.71
min         412.31
25%       9,491.34
50%      18,014.10
75%      32,336.01
max     788,479.36
Name: freight_cost, dtype: float64

In [47]:
fig = px.histogram(
    df,
    x="freight_cost",
    nbins=40,
    template="plotly_white"
)

fig.add_vline(
    x=df["freight_cost"].mean(),
    line_dash="dash",
    line_color="red",
    annotation_text="Mean"
)

fig.add_vline(
    x=df["freight_cost"].median(),
    line_dash="dash",
    line_color="green",
    annotation_text="Median"
)

fig.show()

In [48]:
fig = px.box(

    df,

    y="freight_cost",

    title="Freight Cost Boxplot",

    template="plotly_white"

)

fig.show()

Freight cost exhibits an extremely right-skewed distribution (skewness = 5.88), with the mean ($33.7K) substantially exceeding the median ($18.0K). This indicates that while most shipments incur relatively modest transportation costs, a small number of shipments are considerably more expensive and pull the average upward. The very high kurtosis (41.20) suggests a heavy-tailed distribution with substantially more extreme values than expected under a normal distribution.

## TRansit Days

In [49]:
completed = df.dropna(subset=["transit_days"])

In [50]:
completed["transit_days"].describe()

count   3,455.00
mean        5.33
std         3.05
min        -3.00
25%         3.00
50%         5.00
75%         8.00
max        10.00
Name: transit_days, dtype: float64

In [51]:
fig = px.histogram(

    completed,

    x="transit_days",

    nbins=30,

    title="Transit Time Distribution",

    template="plotly_white"

)

fig.show()

In [52]:
fig = px.box(

    completed,

    y="transit_days",

    title="Transit Time Boxplot",

    template="plotly_white"

)

fig.show()

In [84]:
negative_transit = df[df["transit_days"] < 0]

negative_transit[
    [
        "shipment_id",
        "pickup_date",
        "actual_delivery_date",
        "invalid_date_sequence"
    ]
]

negative_transit[negative_transit["invalid_date_sequence"]==True].count()

shipment_id                72
booking_date               72
pickup_date                72
delivery_date              72
origin_city                72
destination_city           72
region                     72
mode                       72
carrier_id                 72
customer_id                72
promised_delivery_date     72
actual_delivery_date       72
freight_cost               72
distance_km                72
status                     72
missing_booking_date       72
missing_pickup_date        72
missing_actual_delivery    72
invalid_date_sequence      72
delivery_delay_days        72
transit_days               72
booking_to_pickup_days     72
cost_per_km                72
delivery_performance       72
on_time_flag               72
analysis_ready             72
delay_category             72
distance_band              72
dtype: int64

## Delivery Delay

In [53]:
delay_df = df.dropna(subset=["delivery_delay_days"])

In [54]:
delay_df["delivery_delay_days"].describe()

count   3,518.00
mean        0.28
std         3.65
min       -11.00
25%        -2.00
50%         0.00
75%         3.00
max         8.00
Name: delivery_delay_days, dtype: float64

In [73]:
fig = px.histogram(

    delay_df,

    x="delivery_delay_days",

    nbins=35,

    title="Delivery Delay Distribution",

    template="plotly_white"

)

fig.show()

In [55]:
fig = px.box(

    delay_df,

    y="delivery_delay_days",

    title="Delivery Delay Boxplot",

    template="plotly_white"

)

fig.show()

Overall delivery performance is centered around the promised delivery date, with the average shipment arriving approximately 0.3 days later than promised. The median delay of zero days indicates that half of completed shipments arrive on or before the promised delivery date.

## cost per KM

In [58]:
## cost per km
df["cost_per_km"].describe()

count   5,000.00
mean       26.27
std        41.79
min         6.80
25%        10.88
50%        13.29
75%        25.22
max       335.78
Name: cost_per_km, dtype: float64

In [86]:
fig = px.histogram(

    df,

    x="cost_per_km",

    nbins=120,

    title="Distribution of Cost per KM",

    template="plotly_white"

)

fig.update_layout(

    xaxis_title="Cost per KM",

    yaxis_title="Number of Shipments"

)

fig.show()

In [75]:
fig = px.box(

    df,

    y="cost_per_km",

    title="Freight Cost per KM Boxplot",

    template="plotly_white"

)

fig.show()

## Lead time

In [71]:
leadtime = df.dropna(subset=['booking_to_pickup_days'])
leadtime['booking_to_pickup_days'].describe()

count   4,845.00
mean        1.52
std         1.12
min         0.00
25%         1.00
50%         2.00
75%         3.00
max         3.00
Name: booking_to_pickup_days, dtype: float64

In [78]:
fig = px.histogram(

    leadtime,

    x="booking_to_pickup_days",

    nbins=30,

    title="Leadtime Distribution",

    template="plotly_white"

)

fig.show()

In [72]:
fig = px.box(

    leadtime,

    y="booking_to_pickup_days",

    title="Leadtime Boxplot",

    template="plotly_white"

)

fig.show()

Booking-to-pickup lead times are tightly controlled, with nearly all shipments scheduled within three days of booking. The small standard deviation and low variability suggest a highly standardized operational planning process.

In [69]:
distribution_summary = summary_df[
    [
        "Variable",
        "Mean",
        "Median",
        "Std Dev",
        "CV (%)",
        "Skewness",
        "Kurtosis"
    ]
]

distribution_summary

,Variable,Mean,Median,Std Dev,CV (%),Skewness,Kurtosis
0,distance_km,"1,279.76","1,280.00",711.88,55.63,0.00,-1.22
1,freight_cost,"33,741.78","18,014.10","65,530.71",194.21,5.88,41.20
2,transit_days,5.33,5.00,3.05,57.34,-0.14,-0.82
3,delivery_delay_days,0.28,0.00,3.65,"1,291.68",-0.14,-0.50
4,booking_to_pickup_days,1.52,2.00,1.12,73.87,-0.02,-1.37
5,cost_per_km,26.27,13.29,41.79,159.08,4.58,22.32


# shipment status exploration

The dataset contains two shipment outcome variables:

status (original operational status from the source system)
delivery_performance (engineered during the cleaning phase)

Although these variables appear similar, they represent different business concepts.

This section investigates how they differ and validates that delivery_performance is the appropriate measure for SLA and on-time performance analysis.

In [89]:
# Reusable Categorical Analysis Function

def analyze_category(df, column, title, top_n=None):

    summary = (
        df[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="Count")
    )

    summary["Percentage"] = (
        summary["Count"] / len(df) * 100
    ).round(2)

    if top_n:
        plot_df = summary.head(top_n)
    else:
        plot_df = summary

    fig = px.bar(
        plot_df,
        x=column,
        y="Count",
        text="Percentage",
        title=title,
        template="plotly_white"
    )

    fig.update_traces(
        texttemplate="%{text:.2f}%",
        textposition="outside"
    )

    fig.update_layout(
        xaxis_title="",
        yaxis_title="Number of Shipments"
    )

    fig.show()

    return summary

In [90]:
status_summary = analyze_category(
    df,
    column="status",
    title="Operational Shipment Status Distribution"
)

status_summary

,status,Count,Percentage
0,Delivered,3607,72.14
1,Delayed,593,11.86
2,In-Transit,499,9.98
3,Cancelled,301,6.02


In [91]:
performance_summary = analyze_category(
    df,
    column="delivery_performance",
    title="Delivery Performance Distribution"
)

performance_summary

,delivery_performance,Count,Percentage
0,On Time,1795,35.90
1,Late,1723,34.46
2,Unknown,682,13.64
3,In Transit,499,9.98
4,Cancelled,301,6.02


Among all shipments:

35.90% were delivered on time.
34.46% were delivered late.
13.64% could not be evaluated because delivery timing was unavailable despite the shipment being operationally complete.
The remaining shipments were still in transit or cancelled.

In [92]:
comparison = pd.crosstab(
    df["status"],
    df["delivery_performance"],
    margins=True
)

comparison

delivery_performance,Cancelled,In Transit,Late,On Time,Unknown,All
status,,,,,,
Cancelled,301,0,0,0,0,301
Delayed,0,0,239,258,96,593
Delivered,0,0,1484,1537,586,3607
In-Transit,0,499,0,0,0,499
All,301,499,1723,1795,682,5000


In [93]:
comparison_pct = (
    pd.crosstab(
        df["status"],
        df["delivery_performance"],
        normalize="index"
    ) * 100
).round(2)

comparison_pct

delivery_performance,Cancelled,In Transit,Late,On Time,Unknown
status,,,,,
Cancelled,100.00,0.00,0.00,0.00,0.00
Delayed,0.00,0.00,40.30,43.51,16.19
Delivered,0.00,0.00,41.14,42.61,16.25
In-Transit,0.00,100.00,0.00,0.00,0.00


In [94]:
fig = px.imshow(
    comparison_pct,
    text_auto=".1f",
    color_continuous_scale="Blues",
    title="Relationship Between Operational Status and Delivery Performance",
    aspect="auto"
)

fig.update_layout(
    template="plotly_white",
    coloraxis_colorbar_title="%"
)

fig.show()

In [95]:
operational_summary = pd.DataFrame({

    "Metric":[

        "Completed Shipments",

        "Cancelled",

        "In Transit",

        "Analysis Ready"

    ],

    "Count":[

        df["actual_delivery_date"].notna().sum(),

        (df["status"]=="Cancelled").sum(),

        (df["status"]=="In-Transit").sum(),

        df["analysis_ready"].sum()

    ]

})

operational_summary["Percentage"] = (

    operational_summary["Count"]

    / len(df)

    *100

).round(2)

operational_summary

,Metric,Count,Percentage
0,Completed Shipments,3518,70.36
1,Cancelled,301,6.02
2,In Transit,499,9.98
3,Analysis Ready,3444,68.88


The difference between Completed Shipments (70.36%) and Analysis Ready (68.88%) is only 74 shipments, which corresponds exactly to the records flagged with invalid chronological sequences during the data cleaning phase.

In [96]:
eligible = df[df["analysis_ready"]]

on_time_rate = (
    eligible["on_time_flag"]
    .mean()
    *100
)

late_rate = 100 - on_time_rate

summary = pd.DataFrame({

    "Metric":[

        "On-Time Rate",

        "Late Rate"

    ],

    "Percentage":[

        round(on_time_rate,2),

        round(late_rate,2)

    ]

})

summary

,Metric,Percentage
0,On-Time Rate,49.97
1,Late Rate,50.03


Among the 3,444 analysis-ready shipments, the overall delivery performance is almost perfectly balanced: This is a particularly interesting baseline.
the dataset suggests that approximately half of all measurable shipments met their promised delivery dates while the other half did not.

This balance implies that operational performance is likely influenced by underlying factors—such as carrier, customer, region or shipment characteristics

In [97]:
mode_performance = (
    pd.crosstab(
        df["mode"],
        df["delivery_performance"],
        normalize="index"
    ) * 100
).round(2)

mode_performance

delivery_performance,Cancelled,In Transit,Late,On Time,Unknown
mode,,,,,
FTL,6.25,9.54,34.78,36.74,12.69
LTL,5.80,10.04,34.71,34.76,14.68
PTL,5.99,10.76,33.30,36.45,13.50


In [98]:
status_perf = (
    pd.crosstab(
        df["status"],
        df["delivery_performance"],
        normalize="index"
    ) * 100
).round(2)

status_perf = status_perf.reset_index()

fig = px.bar(
    status_perf,
    x="status",
    y=[col for col in status_perf.columns if col != "status"],
    title="Delivery Performance Within Each Operational Status",
    labels={"value": "Percentage", "variable": "Delivery Performance"},
    template="plotly_white"
)

fig.update_layout(
    barmode="stack",
    yaxis_title="Percentage of Shipments"
)

fig.show()

Almost half of the shipments labelled Delivered were actually delivered late.

This means:

"Delivered" only confirms that a shipment reached its destination—it does not indicate whether the delivery met the promised service level.

A substantial proportion of shipments operationally labelled as Delayed were classified as On Time according to the promised delivery date.

This strongly suggests that the operational status field is maintained independently of the SLA calculation. It may reflect internal operational events, exceptions, or manual workflow states rather than actual delivery performance.

These categories map exactly as expected:

All In-Transit shipments remain classified as In Transit.
All Cancelled shipments remain classified as Cancelled.

This provides additional confidence that the engineered logic correctly preserves operational states where no delivery evaluation is possible.

# Exploratory Group Analysis

In [124]:
# ============================================================
# Analysis Populations
# ============================================================

# Entire cleaned dataset
operational_df = df.copy()

# Only shipments suitable for SLA / delivery performance analysis
performance_df = df[df["analysis_ready"]].copy()

print("=" * 50)
print("Analysis Populations")
print("=" * 50)

print(f"Operational Dataset : {len(operational_df):,} shipments")
print(f"Performance Dataset : {len(performance_df):,} shipments")

print(f"\nOperational Coverage : {len(operational_df)/len(df)*100:.1f}%")
print(f"Performance Coverage : {len(performance_df)/len(df)*100:.1f}%")


Analysis Populations
Operational Dataset : 5,000 shipments
Performance Dataset : 3,444 shipments

Operational Coverage : 100.0%
Performance Coverage : 68.9%


In [125]:
# ============================================================
# Group Summary
# ============================================================

def summarize_group(
    operational_df,
    performance_df,
    group_col
):
    """
    Creates a grouped summary using:
    - Operational dataset for workload metrics
    - Performance dataset for SLA metrics
    """

    # -----------------------
    # Operational Metrics
    # -----------------------

    operational = (
        operational_df
        .groupby(group_col)
        .agg(
            Shipments=("shipment_id","count"),
            Avg_Distance=("distance_km","mean"),
            Avg_Freight=("freight_cost","mean"),
            Avg_Cost_per_KM=("cost_per_km","mean")
        )
    )

    operational["Shipment %"] = (
        operational["Shipments"]
        / len(operational_df)
        *100
    )

    # -----------------------
    # Performance Metrics
    # -----------------------

    performance = (
        performance_df
        .groupby(group_col)
        .agg(
            Avg_Transit=("transit_days","mean"),
            Avg_Delay=("delivery_delay_days","mean"),
            On_Time_Rate=("on_time_flag","mean")
        )
    )

    performance["On_Time_Rate"] *=100

    summary = (
        operational
        .join(performance)
        .reset_index()
        .sort_values(
            "Shipments",
            ascending=False
        )
    )

    return summary.round(2)

In [126]:
# ============================================================
# Bar Chart
# ============================================================

def plot_bar(
    summary,
    x,
    y,
    title,
    sort=False
):

    plot_df = summary.copy()

    if sort:

        plot_df = plot_df.sort_values(
            y,
            ascending=False
        )

    fig = px.bar(

        plot_df,

        x=x,

        y=y,

        text=y,

        title=title,

        template="plotly_white"

    )

    fig.update_traces(

        textposition="outside"

    )

    fig.update_layout(

        xaxis_title="",

        yaxis_title=y

    )

    fig.show()

In [127]:
# ============================================================
# Heatmap
# ============================================================

def plot_heatmap(
    data,
    title
):

    fig = px.imshow(

        data,

        text_auto=".1f",

        aspect="auto",

        color_continuous_scale="Blues",

        title=title

    )

    fig.update_layout(

        template="plotly_white"

    )

    fig.show()

In [128]:
# ============================================================
# Scatter Plot
# ============================================================

def plot_scatter(
    data,
    x,
    y,
    color=None,
    title=""
):

    fig = px.scatter(

        data,

        x=x,

        y=y,

        color=color,

        trendline="ols",

        opacity=0.65,

        template="plotly_white",

        title=title

    )

    fig.show()

In [129]:
# ============================================================
# Correlation Matrix
# ============================================================

def correlation_matrix(
    data,
    columns
):

    corr = data[columns].corr()

    plot_heatmap(
        corr,
        "Correlation Matrix"
    )

    return corr

## Region Analysis

In [130]:
# region_summary = summarize_group(df, "region")
region_summary = summarize_group(
    operational_df,
    performance_df,
    "region"
)
region_summary

,region,Shipments,Avg_Distance,Avg_Freight,Avg_Cost_per_KM,Shipment %,Avg_Transit,Avg_Delay,On_Time_Rate
4,West,1049,"1,238.98","32,597.58",25.15,20.98,5.46,0.27,51.29
0,Central,1001,"1,309.86","29,373.61",23.04,20.02,5.52,0.55,48.33
2,North,993,"1,259.28","33,013.87",26.22,19.86,5.56,0.57,49.57
1,East,986,"1,309.12","36,540.11",29.16,19.72,5.42,0.42,50.31
3,South,971,"1,283.93","37,383.86",27.92,19.42,5.33,0.19,52.42


In [131]:
plot_group_bar(

    region_summary,

    x="region",

    y="Shipments",

    title="Shipment Volume by Region"

)

In [132]:
plot_group_bar(

    region_summary,

    x="region",

    y="Avg_Distance",

    title="Average Shipment Distance by Region"

)

In [133]:
plot_group_bar(

    region_summary,

    x="region",

    y="Avg_Freight",

    title="Average Freight Cost by Region"

)

In [134]:
plot_group_bar(

    region_summary,

    x="region",

    y="On_Time_Rate",

    title="Preliminary On-Time Rate by Region"

)

In [135]:
carrier_summary = summarize_group(
    operational_df,
    performance_df,
    "carrier_id"
)

carrier_summary.head(15)

,carrier_id,Shipments,Avg_Distance,Avg_Freight,Avg_Cost_per_KM,Shipment %,Avg_Transit,Avg_Delay,On_Time_Rate
10,CARR_11,362,"1,287.26","21,814.51",16.75,7.24,5.31,0.17,55.25
14,CARR_15,361,"1,295.63","21,720.07",16.73,7.22,5.40,0.34,54.32
1,CARR_02,360,"1,242.73","20,502.91",16.54,7.20,5.93,0.92,40.42
0,CARR_01,352,"1,294.23","21,854.00",16.99,7.04,5.27,0.17,50.81
8,CARR_09,344,"1,267.72","20,706.64",16.61,6.88,5.20,0.19,53.04
6,CARR_07,342,"1,281.56","206,160.68",159.16,6.84,5.79,0.93,43.89
9,CARR_10,330,"1,248.82","20,400.40",16.63,6.60,5.41,0.44,52.59
4,CARR_05,329,"1,325.88","22,641.42",16.94,6.58,5.80,0.73,50.21
13,CARR_14,323,"1,296.26","20,631.25",16.04,6.46,5.26,0.07,54.34
2,CARR_03,321,"1,200.51","19,737.88",16.29,6.42,5.16,0.22,50.00


In [136]:
plot_group_bar(

    carrier_summary.head(15),

    x="carrier_id",

    y="Shipments",

    title="Top 15 Carriers by Shipment Volume"

)

In [138]:
plot_group_bar(

    carrier_summary.head(15),

    x="carrier_id",

    y="Avg_Freight",

    title="Average Freight Cost by Carrier"

)

In [139]:
plot_group_bar(

    carrier_summary.head(15),

    x="carrier_id",

    y="Avg_Delay",

    title="Average Delivery Delay by Carrier"

)

In [140]:
plot_group_bar(

    carrier_summary.head(15),

    x="carrier_id",

    y="On_Time_Rate",

    title="On-Time Rate by Carrier"

)

In [142]:
customer_summary = summarize_group(
    operational_df,
    performance_df,
    "customer_id"
)

customer_summary.head(20)

,customer_id,Shipments,Avg_Distance,Avg_Freight,Avg_Cost_per_KM,Shipment %,Avg_Transit,Avg_Delay,On_Time_Rate
47,CUST_048,60,"1,354.62","40,917.87",28.44,1.20,4.51,-0.77,74.36
84,CUST_085,60,"1,447.27","51,941.93",35.26,1.20,5.61,0.55,47.73
59,CUST_060,55,"1,337.69","33,709.65",23.60,1.10,5.94,0.33,51.52
46,CUST_047,55,"1,448.55","35,195.40",22.02,1.10,4.84,0.30,51.52
29,CUST_030,53,"1,307.13","45,847.86",35.51,1.06,5.85,0.93,41.46
17,CUST_018,52,"1,202.75","37,933.97",28.62,1.04,5.52,-0.32,61.76
55,CUST_056,52,"1,390.44","30,787.04",20.29,1.04,4.56,-0.79,65.52
6,CUST_007,52,"1,441.15","46,874.83",28.34,1.04,5.20,0.57,42.86
88,CUST_089,52,"1,165.19","19,399.43",17.63,1.04,5.41,0.37,50.00
86,CUST_087,51,"1,153.16","31,707.91",28.70,1.02,5.53,0.75,44.44


In [147]:
plot_group_bar(

    customer_summary.head(20),

    x="customer_id",

    y="Shipments",

    title="Top 20 Customers by Shipment Volume"

)

In [148]:
plot_group_bar(

    customer_summary.head(20),

    x="customer_id",

    y="Avg_Delay",

    title="Average Delay by Customer"

)

In [149]:
plot_group_bar(

    customer_summary.head(20),

    x="customer_id",

    y="On_Time_Rate",

    title="On-Time Rate by Customer"

)

In [150]:
mode_summary = summarize_group(
    operational_df,
    performance_df,
    "mode"
)

mode_summary

,mode,Shipments,Avg_Distance,Avg_Freight,Avg_Cost_per_KM,Shipment %,Avg_Transit,Avg_Delay,On_Time_Rate
0,FTL,2033,"1,277.07","51,024.92",39.33,40.66,5.51,0.46,50.28
1,LTL,1982,"1,263.53","24,146.94",19.30,39.64,5.48,0.41,49.11
2,PTL,985,"1,317.98","17,376.65",13.33,19.70,5.44,0.47,51.04


In [151]:
plot_group_bar(

    mode_summary,

    x="mode",

    y="Shipments",

    title="Shipment Volume by Transport Mode"

)

In [152]:
plot_group_bar(

    mode_summary,

    x="mode",

    y="Avg_Delay",

    title="Average Delivery Delay by Transport Mode"

)

In [154]:
distance_summary = summarize_group(
    operational_df,
    performance_df,
    "distance_band"
)

distance_summary

,distance_band,Shipments,Avg_Distance,Avg_Freight,Avg_Cost_per_KM,Shipment %,Avg_Transit,Avg_Delay,On_Time_Rate
4,>2000 km,1059,"2,251.51","61,040.75",27.06,21.18,5.37,0.36,51.90
3,501–1000 km,1046,746.30,"18,929.34",25.34,20.92,5.44,0.40,48.26
2,1501–2000 km,1002,"1,747.83","45,357.32",25.91,20.04,5.59,0.47,50.37
1,1001–1500 km,980,"1,253.51","32,769.20",26.15,19.60,5.58,0.60,48.34
0,0–500 km,913,278.27,"7,343.69",26.92,18.26,5.45,0.40,51.04


In [155]:
plot_group_bar(

    distance_summary,

    x="distance_band",

    y="Shipments",

    title="Shipment Volume by Distance Band"

)

In [156]:
plot_group_bar(

    distance_summary,

    x="distance_band",

    y="Avg_Freight",

    title="Average Freight Cost by Distance Band"

)

In [157]:
plot_group_bar(

    distance_summary,

    x="distance_band",

    y="Avg_Delay",

    title="Average Delivery Delay by Distance Band"

)

# Time series Analysis

In [158]:
# ============================================================
# Create Time Features
# ============================================================

for dataset in [operational_df, performance_df]:

    dataset["booking_year"] = dataset["booking_date"].dt.year
    dataset["booking_month"] = dataset["booking_date"].dt.month
    dataset["booking_month_name"] = dataset["booking_date"].dt.strftime("%b")

    dataset["booking_year_month"] = (
        dataset["booking_date"]
        .dt.to_period("M")
        .astype(str)
    )

    dataset["booking_week"] = (
        dataset["booking_date"]
        .dt.to_period("W")
        .astype(str)
    )

In [159]:
# ============================================================
# Time Series Plot
# ============================================================

def plot_time_series(
    data,
    x,
    y,
    title,
    markers=True
):

    fig = px.line(
        data,
        x=x,
        y=y,
        markers=markers,
        template="plotly_white",
        title=title
    )

    fig.update_layout(
        xaxis_title="",
        yaxis_title=y
    )

    fig.show()

In [160]:
# ============================================================
# Monthly Shipment Volume
# ============================================================

monthly_shipments = (
    operational_df
    .groupby("booking_year_month")
    .size()
    .reset_index(name="Shipments")
)

monthly_shipments

,booking_year_month,Shipments
0,2026-01,827
1,2026-02,759
2,2026-03,847
3,2026-04,798
4,2026-05,878
5,2026-06,820


The busiest month (May) processed only around 16% more shipments than the quietest month (February), suggesting that the freight network experiences relatively stable demand rather than pronounced seasonal peaks.

No evidence of sustained growth or decline is observed across the analysis period.

In [161]:
plot_time_series(
    monthly_shipments,
    "booking_year_month",
    "Shipments",
    "Monthly Shipment Volume"
)

In [162]:
# ============================================================
# Monthly On-Time Rate
# ============================================================

monthly_performance = (
    performance_df
    .groupby("booking_year_month")
    .agg(
        Shipments=("shipment_id","count"),
        On_Time_Rate=("on_time_flag","mean")
    )
    .reset_index()
)

monthly_performance["On_Time_Rate"] *= 100

monthly_performance

,booking_year_month,Shipments,On_Time_Rate
0,2026-01,568,48.77
1,2026-02,530,51.32
2,2026-03,602,48.84
3,2026-04,568,51.76
4,2026-05,590,49.15
5,2026-06,546,50.18


In [163]:
plot_time_series(
    monthly_performance,
    "booking_year_month",
    "On_Time_Rate",
    "Monthly On-Time Rate"
)

Delivery performance remains highly stable throughout the observation period.

Monthly on-time performance fluctuates within a narrow range of approximately 49–52%, with no month exhibiting a sustained deterioration or exceptional improvement.

The highest monthly performance occurs in April (51.76%), while January and March record slightly lower performance at around 49%, but these differences are relatively small.

The overall operational picture suggests a consistently performing logistics network rather than one experiencing periodic service failures.

In [164]:
# ============================================================
# Monthly Average Delay
# ============================================================

monthly_delay = (
    performance_df
    .groupby("booking_year_month")
    .agg(
        Avg_Delay=("delivery_delay_days","mean")
    )
    .reset_index()
)

monthly_delay

,booking_year_month,Avg_Delay
0,2026-01,0.29
1,2026-02,0.50
2,2026-03,0.52
3,2026-04,0.40
4,2026-05,0.52
5,2026-06,0.44


In [165]:
plot_time_series(
    monthly_delay,
    "booking_year_month",
    "Avg_Delay",
    "Monthly Average Delivery Delay"
)

Average delivery delay remains consistently close to half a day throughout the study period.

Although March and May show slightly higher average delays, the differences are modest (approximately 0.2 days) and do not indicate a meaningful operational deterioration.

The delivery process therefore appears operationally stable over time.

In [166]:
monthly_transit = (
    performance_df
    .groupby("booking_year_month")
    .agg(
        Avg_Transit=("transit_days","mean")
    )
    .reset_index()
)

plot_time_series(
    monthly_transit,
    "booking_year_month",
    "Avg_Transit",
    "Monthly Average Transit Time"
)

Transit time is exceptionally stable.

Across all months:

average transit time remains between 5.37 and 5.67 days,
the total variation is only 0.30 days.

Such limited variation suggests that transportation speed is largely consistent across the network.

In [167]:
weekly_metrics = (
    performance_df
    .groupby("booking_week")
    .agg(
        Shipments=("shipment_id","count"),
        Avg_Delay=("delivery_delay_days","mean"),
        On_Time_Rate=("on_time_flag","mean"),
        Avg_Transit=("transit_days","mean")
    )
    .reset_index()
)

weekly_metrics["On_Time_Rate"] *= 100

weekly_metrics.head()

,booking_week,Shipments,Avg_Delay,On_Time_Rate,Avg_Transit
0,2025-12-29/2026-01-04,84,0.45,50.00,5.58
1,2026-01-05/2026-01-11,116,0.49,45.69,5.46
2,2026-01-12/2026-01-18,127,0.06,51.18,5.34
3,2026-01-19/2026-01-25,125,0.50,46.40,5.81
4,2026-01-26/2026-02-01,136,0.06,51.47,5.24


In [168]:
plot_time_series(
    weekly_metrics,
    "booking_week",
    "On_Time_Rate",
    "Weekly On-Time Rate"
)

Weekly on-time performance fluctuates much more noticeably than monthly averages.

For example:

lowest week ≈ 41%
highest week ≈ 64%

These swings are substantially larger than the monthly variation.

In [169]:
plot_time_series(
    weekly_metrics,
    "booking_week",
    "Avg_Delay",
    "Weekly Average Delay"
)

Operations managers would likely monitor performance weekly rather than monthly because weekly reporting is more sensitive to emerging problems.

In [170]:
weekly_metrics["Rolling_On_Time"] = (
    weekly_metrics["On_Time_Rate"]
    .rolling(window=4, min_periods=1)
    .mean()
)

plot_time_series(
    weekly_metrics,
    "booking_week",
    "Rolling_On_Time",
    "4-Week Rolling On-Time Rate"
)

The rolling on-time rate remains close to 50% throughout the study period.

In [174]:
# ============================================================
# Rolling Average Delay
# ============================================================

weekly_metrics["Rolling_Delay"] = (
    weekly_metrics["Avg_Delay"]
    .rolling(window=4, min_periods=1)
    .mean()
)

plot_time_series(
    weekly_metrics,
    "booking_week",
    "Rolling_Delay",
    "4-Week Rolling Average Delivery Delay"
)

Likewise, the rolling average delivery delay remains close to 0.4–0.6 days.

Neither metric exhibits a sustained upward or downward trend.

In [171]:
time_summary = pd.DataFrame({

    "Metric": [
        "Months Covered",
        "Weeks Covered",
        "Average Monthly Shipments",
        "Average Weekly Shipments",
        "Average Monthly On-Time Rate",
        "Average Weekly Delay"
    ],

    "Value": [

        monthly_shipments.shape[0],

        weekly_metrics.shape[0],

        round(monthly_shipments["Shipments"].mean(),2),

        round(weekly_metrics["Shipments"].mean(),2),

        round(monthly_performance["On_Time_Rate"].mean(),2),

        round(weekly_metrics["Avg_Delay"].mean(),2)

    ]
})

time_summary

,Metric,Value
0,Months Covered,6.00
1,Weeks Covered,27.00
2,Average Monthly Shipments,821.50
3,Average Weekly Shipments,126.07
4,Average Monthly On-Time Rate,50.00
5,Average Weekly Delay,0.41


In [172]:
# ============================================================
# Monthly Delivery Performance Mix
# ============================================================

monthly_performance_mix = (
    operational_df
    .groupby(["booking_year_month", "delivery_performance"])
    .size()
    .reset_index(name="Shipments")
)

# Convert counts to percentages within each month
monthly_performance_mix["Percentage"] = (
    monthly_performance_mix.groupby("booking_year_month")["Shipments"]
    .transform(lambda x: x / x.sum() * 100)
)

fig = px.bar(
    monthly_performance_mix,
    x="booking_year_month",
    y="Percentage",
    color="delivery_performance",
    title="Monthly Distribution of Delivery Performance",
    template="plotly_white"
)

fig.update_layout(
    barmode="stack",
    yaxis_title="Percentage of Shipments",
    xaxis_title="Booking Month"
)

fig.show()

In [173]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# ============================================================
# Shipment Volume vs On-Time Rate
# ============================================================

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=monthly_shipments["booking_year_month"],
        y=monthly_shipments["Shipments"],
        name="Shipment Volume"
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=monthly_performance["booking_year_month"],
        y=monthly_performance["On_Time_Rate"],
        mode="lines+markers",
        name="On-Time Rate (%)"
    ),
    secondary_y=True
)

fig.update_layout(
    title="Monthly Shipment Volume vs On-Time Rate",
    template="plotly_white"
)

fig.update_yaxes(
    title_text="Shipments",
    secondary_y=False
)

fig.update_yaxes(
    title_text="On-Time Rate (%)",
    secondary_y=True
)

fig.show()

Shipment volume does not appear to be the primary driver of delivery performance in this dataset.

The logistics network appears capable of maintaining similar service levels despite moderate fluctuations in demand.

In [175]:
# ============================================================
# Monthly KPI Summary
# ============================================================

monthly_kpis = (
    performance_df
    .groupby("booking_year_month")
    .agg(
        Shipments=("shipment_id", "count"),
        On_Time_Rate=("on_time_flag", "mean"),
        Avg_Delay=("delivery_delay_days", "mean"),
        Avg_Transit=("transit_days", "mean"),
        Avg_Freight=("freight_cost", "mean")
    )
    .reset_index()
)

monthly_kpis["On_Time_Rate"] *= 100

monthly_kpis = monthly_kpis.round(2)

monthly_kpis

,booking_year_month,Shipments,On_Time_Rate,Avg_Delay,Avg_Transit,Avg_Freight
0,2026-01,568,48.77,0.29,5.48,"27,867.26"
1,2026-02,530,51.32,0.50,5.51,"34,884.46"
2,2026-03,602,48.84,0.52,5.67,"37,484.97"
3,2026-04,568,51.76,0.40,5.42,"30,131.98"
4,2026-05,590,49.15,0.52,5.45,"32,818.70"
5,2026-06,546,50.18,0.44,5.37,"33,868.71"


In [176]:
# ============================================================
# Week-over-Week Change
# ============================================================

weekly_metrics["WoW_On_Time_Change"] = (
    weekly_metrics["On_Time_Rate"].diff()
)

weekly_metrics["WoW_Delay_Change"] = (
    weekly_metrics["Avg_Delay"].diff()
)

weekly_metrics[
    [
        "booking_week",
        "On_Time_Rate",
        "WoW_On_Time_Change",
        "Avg_Delay",
        "WoW_Delay_Change"
    ]
].head(10)

,booking_week,On_Time_Rate,WoW_On_Time_Change,Avg_Delay,WoW_Delay_Change
0,2025-12-29/2026-01-04,50.00,NaN,0.45,NaN
1,2026-01-05/2026-01-11,45.69,-4.31,0.49,0.04
2,2026-01-12/2026-01-18,51.18,5.49,0.06,-0.43
3,2026-01-19/2026-01-25,46.40,-4.78,0.50,0.43
4,2026-01-26/2026-02-01,51.47,5.07,0.06,-0.44
5,2026-02-02/2026-02-08,54.35,2.88,0.22,0.16
6,2026-02-09/2026-02-15,52.46,-1.89,0.46,0.24
7,2026-02-16/2026-02-22,43.06,-9.40,0.97,0.51
8,2026-02-23/2026-03-01,53.66,10.60,0.42,-0.54
9,2026-03-02/2026-03-08,50.40,-3.26,0.46,0.03


In [177]:
# ============================================================
# Monthly Delay Distribution
# ============================================================

fig = px.box(
    performance_df,
    x="booking_month_name",
    y="delivery_delay_days",
    title="Monthly Distribution of Delivery Delays",
    template="plotly_white"
)

fig.update_layout(
    xaxis_title="Booking Month",
    yaxis_title="Delivery Delay (Days)"
)

fig.show()

In [178]:
# ============================================================
# Reusable Time Aggregation Function
# ============================================================

def summarize_time(
    data,
    time_col
):
    """
    Summarize operational and performance metrics over time.
    """

    summary = (
        data.groupby(time_col)
        .agg(
            Shipments=("shipment_id", "count"),
            Avg_Freight=("freight_cost", "mean"),
            Avg_Delay=("delivery_delay_days", "mean"),
            Avg_Transit=("transit_days", "mean"),
            On_Time_Rate=("on_time_flag", "mean")
        )
        .reset_index()
    )

    summary["On_Time_Rate"] *= 100

    return summary.round(2)

# Regional Delivery Performance

In [179]:
# ============================================================
# Regional KPI Summary
# ============================================================

def regional_summary(data):

    summary = (
        data
        .groupby("region")
        .agg(
            Shipments=("shipment_id","count"),

            On_Time_Rate=("on_time_flag","mean"),

            Avg_Delay=("delivery_delay_days","mean"),

            Avg_Transit=("transit_days","mean"),

            Avg_Distance=("distance_km","mean"),

            Avg_Freight=("freight_cost","mean"),

            Avg_Cost_per_KM=("cost_per_km","mean")
        )
        .reset_index()
    )

    summary["On_Time_Rate"] *= 100

    summary = summary.sort_values(
        "On_Time_Rate"
    )

    return summary.round(2)

regional_kpis = regional_summary(performance_df)

regional_kpis

,region,Shipments,On_Time_Rate,Avg_Delay,Avg_Transit,Avg_Distance,Avg_Freight,Avg_Cost_per_KM
0,Central,836,48.33,0.55,5.52,"1,301.20","28,903.27",22.54
2,North,811,49.57,0.57,5.56,"1,252.28","33,156.62",26.36
1,East,819,50.31,0.42,5.42,"1,301.79","37,095.95",29.91
4,West,854,51.29,0.27,5.46,"1,250.54","31,039.26",23.95
3,South,124,52.42,0.19,5.33,"1,315.21","39,301.05",24.94


In [180]:
fig = px.bar(

    regional_kpis,

    x="region",

    y="On_Time_Rate",

    color="On_Time_Rate",

    text="On_Time_Rate",

    title="Regional On-Time Delivery Performance",

    template="plotly_white"

)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

In [182]:
regional_long = regional_kpis.melt(

    id_vars="region",

    value_vars=[

        "Avg_Delay",

        "Avg_Transit",

        "Avg_Distance",

        "Avg_Cost_per_KM"

    ],

    var_name="Metric",

    value_name="Value"

)

fig = px.bar(

    regional_long,

    x="region",

    y="Value",

    facet_col="Metric",

    facet_col_wrap=2,

    color="Metric",

    template="plotly_white",

    title="Regional Operational Metrics"
)

fig.show()

In [204]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(
    performance_df["region"],
    performance_df["on_time_flag"]
)

chi2, p, dof, expected = chi2_contingency(contingency)

print("Chi-square statistic:", round(chi2, 2))
print("P-value:", p)

Chi-square statistic: 1.88
P-value: 0.7569642548015834


In [189]:
driver_summary = (

    performance_df

    .groupby("region")

    .agg(

        Avg_Distance=("distance_km","mean"),

        Avg_Transit=("transit_days","mean"),

        Avg_Delay=("delivery_delay_days","mean"),

        Avg_Freight=("freight_cost","mean"),

        Avg_Cost_per_KM=("cost_per_km","mean")

    )

    .round(2)

)

driver_summary

,Avg_Distance,Avg_Transit,Avg_Delay,Avg_Freight,Avg_Cost_per_KM
region,,,,,
Central,"1,301.20",5.52,0.55,"28,903.27",22.54
East,"1,301.79",5.42,0.42,"37,095.95",29.91
North,"1,252.28",5.56,0.57,"33,156.62",26.36
South,"1,315.21",5.33,0.19,"39,301.05",24.94
West,"1,250.54",5.46,0.27,"31,039.26",23.95


In [195]:
carrier_region = (

    performance_df

    .groupby(

        ["region","carrier_id"]

    )

    .size()

    .reset_index(name="Shipments")

)

carrier_region.head()

,region,carrier_id,Shipments
0,Central,CARR_01,61
1,Central,CARR_02,69
2,Central,CARR_03,65
3,Central,CARR_04,44
4,Central,CARR_05,58


In [191]:
fig = px.bar(

    carrier_region,

    x="region",

    y="Shipments",

    color="carrier_id",

    title="Carrier Distribution by Region",

    template="plotly_white"

)

fig.show()

In [192]:
fig = px.box(

    performance_df,

    x="region",

    y="distance_km",

    color="region",

    title="Shipment Distance by Region",

    template="plotly_white"

)

fig.show()

In [193]:
fig = px.box(

    performance_df,

    x="region",

    y="delivery_delay_days",

    color="region",

    title="Delivery Delay Distribution by Region",

    template="plotly_white"

)

fig.show()

In [196]:
performance_df.groupby(
    ["region","carrier_id"]
).agg(
    Shipments=("shipment_id","count"),
    On_Time=("on_time_flag","mean"),
    Avg_Delay=("delivery_delay_days","mean")
)

Shipments  On_Time  Avg_Delay
region  carrier_id                               
Central CARR_01            61     0.49       0.30
        CARR_02            69     0.41       0.84
        CARR_03            65     0.46       0.00
        CARR_04            44     0.48       1.36
        CARR_05            58     0.55       0.78
...                       ...      ...        ...
West    CARR_11            52     0.62      -0.35
        CARR_12            52     0.60      -0.38
        CARR_13            56     0.39       1.23
        CARR_14            61     0.52       0.20
        CARR_15            60     0.55       0.23

[75 rows x 3 columns]

In [197]:
performance_df.groupby(
    ["region","distance_band"]
).agg(
    On_Time=("on_time_flag","mean")
)

On_Time
region  distance_band         
Central 0–500 km          0.45
        1001–1500 km      0.47
        1501–2000 km      0.48
        501–1000 km       0.47
        >2000 km          0.54
East    0–500 km          0.51
        1001–1500 km      0.51
        1501–2000 km      0.51
        501–1000 km       0.47
        >2000 km          0.51
North   0–500 km          0.48
        1001–1500 km      0.46
        1501–2000 km      0.52
        501–1000 km       0.51
        >2000 km          0.51
South   0–500 km          0.56
        1001–1500 km      0.65
        1501–2000 km      0.29
        501–1000 km       0.63
        >2000 km          0.48
West    0–500 km          0.58
        1001–1500 km      0.48
        1501–2000 km      0.53
        501–1000 km       0.46
        >2000 km          0.52

In [205]:
from scipy.stats import kruskal

groups = [
    performance_df.loc[
        performance_df["region"] == r,
        "delivery_delay_days"
    ].dropna()
    for r in performance_df["region"].unique()
]

stat, p = kruskal(*groups)

print(stat, p)

4.351506821243864 0.36051975973181216


In [207]:
# ==========================================================
# Helper Function
# Compute Wilson Confidence Interval
# ==========================================================

def wilson_ci(successes, total):

    lower, upper = proportion_confint(
        successes,
        total,
        method="wilson"
    )

    return lower * 100, upper * 100


# ==========================================================
# Helper Function
# National Average
# ==========================================================

def national_on_time_rate(df):

    return df["on_time_flag"].mean() * 100

In [211]:
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import kruskal
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

regional_ci = (

    performance_df

    .groupby("region")

    .agg(
        Shipments=("shipment_id","count"),
        On_Time=("on_time_flag","sum")
    )

    .reset_index()

)

regional_ci["On_Time_Rate"] = (
    regional_ci["On_Time"] /
    regional_ci["Shipments"]
) * 100

lower = []
upper = []

for success, total in zip(
    regional_ci["On_Time"],
    regional_ci["Shipments"]
):

    l, u = wilson_ci(success, total)

    lower.append(l)
    upper.append(u)

regional_ci["CI_Lower"] = lower
regional_ci["CI_Upper"] = upper

regional_ci

,region,Shipments,On_Time,On_Time_Rate,CI_Lower,CI_Upper
0,Central,836,404.00,48.33,44.95,51.71
1,East,819,412.00,50.31,46.89,53.72
2,North,811,402.00,49.57,46.14,53.00
3,South,124,65.00,52.42,43.69,61.00
4,West,854,438.00,51.29,47.94,54.63


In [212]:
fig = go.Figure()

fig.add_trace(

    go.Bar(

        x=regional_ci["region"],

        y=regional_ci["On_Time_Rate"],

        error_y=dict(

            type="data",

            symmetric=False,

            array=regional_ci["CI_Upper"]-regional_ci["On_Time_Rate"],

            arrayminus=regional_ci["On_Time_Rate"]-regional_ci["CI_Lower"]

        ),

        text=regional_ci["On_Time_Rate"].round(1),

        textposition="outside"

    )

)

fig.update_layout(

    title="Regional On-Time Rate (95% Confidence Intervals)",

    xaxis_title="Region",

    yaxis_title="On-Time Rate (%)",

    template="plotly_white"

)

fig.show()

In [213]:
national_avg = national_on_time_rate(performance_df)

regional_ci["Difference"] = (

    regional_ci["On_Time_Rate"] -

    national_avg

)

regional_ci.sort_values("Difference")

,region,Shipments,On_Time,On_Time_Rate,CI_Lower,CI_Upper,Difference
0,Central,836,404.00,48.33,44.95,51.71,-1.65
2,North,811,402.00,49.57,46.14,53.00,-0.40
1,East,819,412.00,50.31,46.89,53.72,0.33
4,West,854,438.00,51.29,47.94,54.63,1.32
3,South,124,65.00,52.42,43.69,61.00,2.45


In [214]:
fig = px.bar(

    regional_ci.sort_values("Difference"),

    x="Difference",

    y="region",

    orientation="h",

    color="Difference",

    color_continuous_scale="RdYlGn",

    text="Difference",

    title="Deviation from National On-Time Rate",

    template="plotly_white"

)

fig.add_vline(

    x=0,

    line_dash="dash",

    line_color="black"

)

fig.update_traces(texttemplate="%{text:.2f}%")

fig.show()

In [215]:
carrier_region = (

    performance_df

    .groupby(

        ["carrier_id","region"]

    )

    .agg(

        Shipments=("shipment_id","count"),

        On_Time=("on_time_flag","mean"),

        Avg_Delay=("delivery_delay_days","mean")

    )

    .reset_index()

)

carrier_region["On_Time"] *= 100

carrier_region.head()

,carrier_id,region,Shipments,On_Time,Avg_Delay
0,CARR_01,Central,61,49.18,0.30
1,CARR_01,East,69,49.28,0.20
2,CARR_01,North,46,54.35,0.39
3,CARR_01,South,10,70.00,-1.30
4,CARR_01,West,60,48.33,0.10


In [216]:
carrier_heatmap = carrier_region.pivot(

    index="carrier_id",

    columns="region",

    values="On_Time"

)

fig = px.imshow(

    carrier_heatmap,

    text_auto=".1f",

    aspect="auto",

    color_continuous_scale="RdYlGn",

    title="Carrier On-Time Rate Across Regions"

)

fig.show()

In [217]:
delay_heatmap = carrier_region.pivot(

    index="carrier_id",

    columns="region",

    values="Avg_Delay"

)

fig = px.imshow(

    delay_heatmap,

    text_auto=".2f",

    color_continuous_scale="RdYlGn_r",

    aspect="auto",

    title="Average Delay by Carrier and Region"

)

fig.show()

In [218]:
distance_region = (

    performance_df

    .groupby(

        ["region","distance_band"]

    )

    .agg(

        On_Time=("on_time_flag","mean")

    )

    .reset_index()

)

distance_region["On_Time"] *= 100

pivot = distance_region.pivot(

    index="region",

    columns="distance_band",

    values="On_Time"

)

fig = px.imshow(

    pivot,

    text_auto=".1f",

    color_continuous_scale="RdYlGn",

    title="Regional On-Time Rate by Distance Band",

    aspect="auto"

)

fig.show()

In [220]:
scatter = (

    performance_df

    .groupby("region")

    .agg(

        Avg_Delay=("delivery_delay_days","mean"),

        On_Time=("on_time_flag","mean")

    )

    .reset_index()

)

scatter["On_Time"] *= 100

fig = px.scatter(

    scatter,

    x="Avg_Delay",

    y="On_Time",

    text="region",

    size="On_Time",

    title="Average Delay vs On-Time Rate",

    template="plotly_white"

)

fig.update_traces(textposition="top center")

fig.show()

In [221]:
groups = [

    performance_df.loc[

        performance_df["region"]==region,

        "delivery_delay_days"

    ].dropna()

    for region in performance_df["region"].unique()

]

stat, p = kruskal(*groups)

print(f"Kruskal Statistic : {stat:.3f}")
print(f"P-value : {p:.4f}")

Kruskal Statistic : 4.352
P-value : 0.3605


In [222]:
regional_summary = (

    performance_df

    .groupby("region")

    .agg(

        Shipments=("shipment_id","count"),

        On_Time=("on_time_flag","mean"),

        Avg_Delay=("delivery_delay_days","mean"),

        Avg_Transit=("transit_days","mean"),

        Avg_Distance=("distance_km","mean"),

        Avg_Freight=("freight_cost","mean")

    )

    .round(2)

)

regional_summary["On_Time"] *= 100

regional_summary["Difference from National"] = (

    regional_summary["On_Time"]

    - national_avg

).round(2)

regional_summary

,Shipments,On_Time,Avg_Delay,Avg_Transit,Avg_Distance,Avg_Freight,Difference from National
region,,,,,,,
Central,836,48.00,0.55,5.52,"1,301.20","28,903.27",-1.97
East,819,50.00,0.42,5.42,"1,301.79","37,095.95",0.03
North,811,50.00,0.57,5.56,"1,252.28","33,156.62",0.03
South,124,52.00,0.19,5.33,"1,315.21","39,301.05",2.03
West,854,51.00,0.27,5.46,"1,250.54","31,039.26",1.03


In [223]:
# ==========================================================
# Carrier Performance vs National Average
# ==========================================================

carrier_region = (
    performance_df
    .groupby(["region", "carrier_id"])
    .agg(
        Shipments=("shipment_id", "count"),
        On_Time=("on_time_flag", "mean"),
        Avg_Delay=("delivery_delay_days", "mean")
    )
    .reset_index()
)

carrier_region["On_Time"] *= 100

national_avg = performance_df["on_time_flag"].mean() * 100

MIN_SAMPLE = 30

carrier_region = carrier_region[
    carrier_region["Shipments"] >= MIN_SAMPLE
].copy()

carrier_region["Above_National"] = np.where(
    carrier_region["On_Time"] >= national_avg,
    "Above National",
    "Below National"
)

fig = px.bar(
    carrier_region,
    x="carrier_id",
    y="On_Time",
    color="Above_National",
    facet_col="region",
    facet_col_wrap=3,
    text="On_Time",
    title="Carrier On-Time Rate by Region vs National Average",
    color_discrete_map={
        "Above National": "#2ca02c",
        "Below National": "#d62728"
    },
    template="plotly_white"
)

fig.add_hline(
    y=national_avg,
    line_dash="dash",
    line_color="black",
    annotation_text=f"National Avg ({national_avg:.1f}%)"
)

fig.update_traces(texttemplate="%{text:.1f}")

fig.update_layout(height=700)

fig.show()

In [225]:
national_delay = performance_df["delivery_delay_days"].mean()

carrier_region["Delay_vs_National"] = (
    carrier_region["Avg_Delay"] - national_delay
)

fig = px.bar(
    carrier_region,
    x="carrier_id",
    y="Delay_vs_National",
    color="Delay_vs_National",
    facet_col="region",
    facet_col_wrap=3,
    text="Delay_vs_National",
    color_continuous_scale="RdYlGn_r",
    title="Carrier Average Delay Relative to National Average",
    template="plotly_white"
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="black"
)

fig.update_traces(
    texttemplate="%{text:.2f}"
)

fig.update_layout(height=700)

fig.show()

In [227]:
carrier_heat = carrier_region.copy()

carrier_heat["Difference"] = (
    carrier_heat["On_Time"] - national_avg
)

heat = carrier_heat.pivot(
    index="carrier_id",
    columns="region",
    values="Difference"
)

fig = px.imshow(
    heat,
    text_auto=".1f",
    color_continuous_scale="RdYlGn",
    aspect="auto",
    title="Carrier On-Time Performance Relative to National Average (%)"
)

fig.update_layout(
    xaxis_title="Region",
    yaxis_title="Carrier"
)

fig.show()

# Freight Cost Analysis

In [229]:
pricing_df = df.copy()

print(f"Pricing Dataset Size: {len(pricing_df):,} shipments")

Pricing Dataset Size: 5,000 shipments


In [230]:
pricing_summary = (
    pricing_df[
        ["distance_km", "freight_cost", "cost_per_km"]
    ]
    .describe()
    .T
)

display(pricing_summary)

,count,mean,std,min,25%,50%,75%,max
distance_km,"5,000.00","1,279.76",711.88,50.00,655.00,"1,280.00","1,901.00","2,499.00"
freight_cost,"5,000.00","33,741.78","65,530.71",412.31,"9,491.34","18,014.10","32,336.01","788,479.36"
cost_per_km,"5,000.00",26.27,41.79,6.80,10.88,13.29,25.22,335.78


In [231]:
from scipy.stats import pearsonr, spearmanr

pearson_r, pearson_p = pearsonr(
    pricing_df["distance_km"],
    pricing_df["freight_cost"]
)

spearman_r, spearman_p = spearmanr(
    pricing_df["distance_km"],
    pricing_df["freight_cost"]
)

correlation_results = pd.DataFrame({
    "Metric": [
        "Pearson Correlation",
        "Pearson p-value",
        "Spearman Correlation",
        "Spearman p-value"
    ],
    "Value": [
        pearson_r,
        pearson_p,
        spearman_r,
        spearman_p
    ]
})

display(correlation_results.round(4))

,Metric,Value
0,Pearson Correlation,0.30
1,Pearson p-value,0.00
2,Spearman Correlation,0.73
3,Spearman p-value,0.00


In [232]:
corr_matrix = pricing_df[
    [
        "distance_km",
        "freight_cost",
        "cost_per_km"
    ]
].corr(method="spearman")

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="Spearman Correlation Matrix",
)

fig.show()

In [237]:
fig = px.scatter(
    pricing_df,
    x="distance_km",
    y="freight_cost",
    opacity=0.45,
    trendline="ols",
    title="Freight Cost vs Distance",
    labels={
        "distance_km": "Distance (km)",
        "freight_cost": "Freight Cost"
    },
    template="plotly_white"
)

fig.update_layout(
    height=600
)

fig.show()

In [234]:
from sklearn.linear_model import LinearRegression

X = pricing_df[["distance_km"]]
y = pricing_df["freight_cost"]

reg = LinearRegression()

reg.fit(X, y)

pricing_df["predicted_cost"] = reg.predict(X)

r2 = reg.score(X, y)

regression_summary = pd.DataFrame({
    "Metric": [
        "Intercept",
        "Slope",
        "R²"
    ],
    "Value": [
        reg.intercept_,
        reg.coef_[0],
        r2
    ]
})

display(regression_summary.round(3))

,Metric,Value
0,Intercept,"-1,078.01"
1,Slope,27.21
2,R²,0.09


In [239]:
relationship_summary = pd.DataFrame({

    "Metric":[

        "Total Shipments",

        "Pearson Correlation",

        "Spearman Correlation",

        "Regression R²",

        "Average Distance",

        "Average Freight"

    ],

    "Value":[

        len(pricing_df),

        pearson_r,

        spearman_r,

        r2,

        pricing_df["distance_km"].mean(),

        pricing_df["freight_cost"].mean()

    ]

})

display(relationship_summary.round(3))

,Metric,Value
0,Total Shipments,"5,000.00"
1,Pearson Correlation,0.30
2,Spearman Correlation,0.73
3,Regression R²,0.09
4,Average Distance,"1,279.76"
5,Average Freight,"33,741.78"
